<a href="https://colab.research.google.com/github/DSCmatter/Fencio_ass/blob/main/training/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QLoRA runner for the red-team attacker model
Run on a free Colab or Kaggle GPU. This notebook never runs on the local machine. It installs training dependencies, trains on the processed dataset, saves checkpoints, evaluates the base versus tuned model, and serves the adapter through a cloudflared tunnel.
The repo URL is set by default. Override it with the `REPO_URL` environment variable if you fork. No Hugging Face account is needed. Qwen2.5-3B-Instruct is a public model.

In [1]:
import os
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/DSCmatter/Fencio_ass.git')
if os.path.isdir('redteam-agent'):
    !git -C redteam-agent fetch origin main
    !git -C redteam-agent reset --hard origin/main
else:
    !git clone -q {REPO_URL} redteam-agent
%cd redteam-agent
!pwd
!git log --oneline -1

/content/redteam-agent
/content/redteam-agent
27f960f (HEAD -> main, origin/main, origin/HEAD) Use plain unsloth install on Kaggle


In [2]:
# pip 26.3 blocks imports during build isolation. Pin below it.
!python -m pip install -q "pip<26.3"
!python -m pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" trl transformers datasets accelerate peft bitsandbytes fastapi uvicorn pyyaml httpx
!python -m pip install -q --no-deps xformers

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [12]:
# Sync to latest main before training
!git -C /content/redteam-agent fetch origin main
!git -C /content/redteam-agent reset --hard origin/main
!grep base_model /content/redteam-agent/configs/train.yaml
!git -C /content/redteam-agent log --oneline -1

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 505 bytes | 42.00 KiB/s, done.
From https://github.com/DSCmatter/Fencio_ass
 * branch            main       -> FETCH_HEAD
   2282c4a..6eee797  main       -> origin/main
HEAD is now at 6eee797 Launch cloudflared by absolute path
base_model: Qwen/Qwen2.5-3B-Instruct
fallback_base_model: Qwen/Qwen2.5-1.5B-Instruct
6eee797 (HEAD -> main, origin/main, origin/HEAD) Launch cloudflared by absolute path


In [4]:
!python training/train_qlora.py --config configs/train.yaml --train-file data/processed/train.jsonl --output checkpoints/qlora-adapter

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
W1126 06:09:51.896000 1452 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1126 06:09:51.936000 1452 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Unsloth: xformers does not load with torch 2.14.1+cu130, using PyTorch attention (UNSLOTH_ENABLE_LOGGING=1 for the fix).
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.12: Fast Qwen2 patching. Transformers: 5.18.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu

In [5]:
import shutil, os
drive_dir = os.environ.get('DRIVE_DIR', '/content/drive/MyDrive/redteam-checkpoints')
os.makedirs(drive_dir, exist_ok=True)
shutil.copytree('checkpoints/qlora-adapter', drive_dir + '/qlora-adapter', dirs_exist_ok=True)
print('checkpoint copied to', drive_dir)

checkpoint copied to /content/drive/MyDrive/redteam-checkpoints


In [10]:
!python training/evaluate_model.py --config configs/train.yaml --test-file data/processed/test.jsonl --adapter checkpoints/qlora-adapter --output eval/results/model_eval.json

config.json: 100% 661/661 [00:00<00:00, 3.20MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 23.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 80.1MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 73.9MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 107MB/s]
W1126 07:01:11.195000 14937 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1126 07:01:11.243000 14937 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
model.safetensors.index.json: 100% 35.6k/35.6k [00:00<00:00, 106MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files

## Remote inference
Run the next cell only when the assessment needs the fine-tuned attacker. It prints a cloudflared URL. Paste that URL into `configs/agent.yaml` as `model.base_url`, or export `RT_MODEL_BASE_URL` before running the local agent.

In [13]:
!python training/serve_notebook.py --config configs/train.yaml --adapter checkpoints/qlora-adapter --port 8000

W1126 07:18:12.292000 19444 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1126 07:18:12.344000 19444 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100% 434/434 [00:12<00:00, 34.22it/s]
TUNNEL_URL=https://configurations-patterns-connectivity-enormous.trycloudflare.com/v1
Paste this URL into configs/agent.yaml as model.base_url
^C
